# Answering Business Questions Using SQL

## Connect to the database

In [2]:
%%capture
%load_ext sql
%sql sqlite:///chinook.db

## Overview of the Data

In [2]:
%%sql
SELECT
    name,
    type
FROM sqlite_master
WHERE type IN ("table","view");

 * sqlite:///chinook.db
Done.


name,type
album,table
artist,table
customer,table
employee,table
genre,table
invoice,table
invoice_line,table
media_type,table
playlist,table
playlist_track,table


## Selecting Albums to Purchase
I need to find which genres sell the best in the USA.

In [3]:
%%sql
WITH track_id_usa AS
        (
        SELECT
            il.invoice_line_id,
            il.track_id,
            il.quantity,
            i.total,
            c.country
        FROM invoice_line il
        INNER JOIN invoice i ON il.invoice_id = i.invoice_id
        INNER JOIN customer c ON i.customer_id = c.customer_id
        WHERE c.country = "USA"
        ),
    
    track_genre AS
        (
        SELECT 
            t.track_id,
            g.name  
        FROM track t
        INNER JOIN genre g ON t.genre_id = g.genre_id
        )
    
SELECT 
    tg.name genre,
    COUNT(tsu.invoice_line_id) tracks_sold,
    ROUND(CAST(COUNT(tsu.invoice_line_id) AS Float) / 
         (SELECT COUNT(*) FROM track_id_usa), 4) AS percentage_sold
FROM track_id_usa tsu
INNER JOIN track_genre tg ON tsu.track_id = tg.track_id
GROUP BY 1
ORDER BY 2 DESC;


 * sqlite:///chinook.db
Done.


genre,tracks_sold,percentage_sold
Rock,561,0.5338
Alternative & Punk,130,0.1237
Metal,124,0.118
R&B/Soul,53,0.0504
Blues,36,0.0343
Alternative,35,0.0333
Pop,22,0.0209
Latin,22,0.0209
Hip Hop/Rap,20,0.019
Jazz,14,0.0133


In [4]:
%%sql
SELECT
    ar.name artist_name,
    a.title album_name,
    g.name genre,
    COUNT(il.invoice_line_id) tracks_sold   
FROM track t
INNER JOIN genre g ON t.genre_id = g.genre_id
INNER JOIN album a ON t.album_id = a.album_id
INNER JOIN artist ar ON a.artist_id = ar.artist_id
INNER JOIN invoice_line il ON t.track_id = il.track_id
GROUP BY ar.name, g.name 
ORDER BY 4 DESC 
LIMIT 3;


 * sqlite:///chinook.db
Done.


artist_name,album_name,genre,tracks_sold
Queen,Greatest Hits II,Rock,192
Jimi Hendrix,Are You Experienced?,Rock,187
Nirvana,From The Muddy Banks Of The Wishkah [live],Rock,130


Based on the tracks sold by genre in USA, I would recommend to buy the following albumns:
1. Greatest Hits II by Queen
2. Are You Experienced? by Jimi Hendrix
3. From The Muddy Banks Of The Wishkah [live] by Nirvana

## Analyzing Employee Sales Performance

In [5]:
%%sql
WITH customer_invoice AS 
    (
    SELECT 
        c.support_rep_id employee_id,
        c.customer_id,
        i.total
    FROM customer c
    INNER JOIN invoice i ON c.customer_id = i.customer_id
    )

SELECT
    e.first_name || " " || e.last_name employee_name,
    e.city,
    e.country,
    e.hire_date, 
    SUM(ci.total) total_sales,
    ROUND(SUM(ci.total) * 100.0 / (SELECT SUM(total) FROM customer_invoice), 4) sales_percentage
FROM customer_invoice ci 
INNER JOIN employee e ON ci.employee_id = e.employee_id
GROUP BY 1;

 * sqlite:///chinook.db
Done.


employee_name,city,country,hire_date,total_sales,sales_percentage
Jane Peacock,Calgary,Canada,2017-04-01 00:00:00,1731.51,36.7669
Margaret Park,Calgary,Canada,2017-05-03 00:00:00,1584.0,33.6346
Steve Johnson,Calgary,Canada,2017-10-17 00:00:00,1393.92,29.5985


Jane Peacock has the highest total sales with approximately 36%. The city and country information shows that all three employees are located in Calgary, Canada. It's worth noting that Jane Peacock was hired first in April 2017, followed by Margaret Park in May 2017, and then Steve Johnson in October 2017.

## Analyzing Sales by Country

In [120]:
%%sql
WITH country_customers AS
    (
    SELECT 
        c.country,
        COUNT(DISTINCT c.customer_id) number_customers,
        SUM(il.unit_price) total_sales,
        ROUND(SUM(il.unit_price) / COUNT(DISTINCT c.customer_id), 4) average_value_customer,
        ROUND(SUM(il.unit_price) / COUNT(DISTINCT il.invoice_id), 4) average_order_value
    FROM customer c
    INNER JOIN invoice i ON c.customer_id = i.customer_id
    INNER JOIN invoice_line il ON i.invoice_id = il.invoice_id
    GROUP BY 1
    
    )
    
SELECT 
    CASE
        WHEN number_customers = 1 THEN "Other"
        ELSE country
    END AS country_other,
    SUM(number_customers) total_customers,
    total_sales,
    average_value_customer,
    average_order_value
FROM country_customers
GROUP BY country_other
ORDER BY number_customers DESC;

 * sqlite:///chinook.db
Done.


country_other,total_customers,total_sales,average_value_customer,average_order_value
USA,13,1040.49,80.0377,7.9427
Canada,8,535.59,66.9488,7.0472
France,5,389.07,77.814,7.7814
Brazil,5,427.68,85.536,7.0111
Germany,4,334.62,83.655,8.1615
United Kingdom,3,245.52,81.84,8.7686
Portugal,2,185.13,92.565,6.3838
India,2,183.15,91.575,8.7214
Czech Republic,2,273.24,136.62,9.108
Other,15,39.6,39.6,7.92


USA has the highest total sales and total customers, indicating a strong market presence. Czech Republic stands out with high average value per customer and average order value, suggesting potential for higher-profit margins.

## Albums vs Individual Tracks

The Chinook Store allows customer to make purchases in two ways: whole album and a collection of one or more individual tracks. \
It is important to check what percentage of purchases correspond individual tracks and whole albums.

In [31]:
%%sql
WITH album_summary AS
    (
    SELECT
        t.album_id,
        CASE
            WHEN COUNT(DISTINCT il.track_id) = 1 THEN 'Track'
            WHEN COUNT(DISTINCT il.track_id) = 2 THEN 'Track'
            ELSE 'Album'
        END AS category,
        COUNT(DISTINCT il.track_id) AS tracks_in_album,
        COUNT(il.invoice_id) number_of_invoices,
        ROUND(COUNT(il.invoice_id) * 100.0 / (SELECT COUNT(*) FROM invoice_line), 4) invoice_percentage,
        COUNT(il.track_id) total_tracks,
        ROUND(SUM(il.quantity * il.unit_price), 2) total_revenue
    FROM invoice_line il
    INNER JOIN track t ON il.track_id = t.track_id
    GROUP BY album_id
    )

SELECT
    category,
    SUM(number_of_invoices) AS number_of_invoices,
    SUM(invoice_percentage) AS invoice_percentage
FROM album_summary
GROUP BY category;


 * sqlite:///chinook.db
Done.


category,number_of_invoices,invoice_percentage
Album,4554,95.7324
Track,203,4.2664


In [37]:
%%sql
WITH album_summary AS
    (
    SELECT
        t.album_id,
        CASE
            WHEN COUNT(DISTINCT il.track_id) = 1 THEN 'Track'
            WHEN COUNT(DISTINCT il.track_id) = 2 THEN 'Track'
            ELSE 'Album'
        END AS category,
        il.invoice_id,
        COUNT(DISTINCT il.track_id) tracks_in_album,
        ROUND(SUM(il.quantity * il.unit_price), 2) total_revenue
    FROM invoice_line il
    INNER JOIN track t ON il.track_id = t.track_id
    GROUP BY album_id, il.invoice_id
    )

SELECT
    category,
    COUNT(DISTINCT invoice_id) number_of_invoices,
    ROUND(COUNT(DISTINCT invoice_id) * 100.0 / (SELECT COUNT(DISTINCT invoice_id) FROM invoice_line), 2) invoice_percentage,
    SUM(total_revenue) total_revenue
FROM album_summary
GROUP BY category;

 * sqlite:///chinook.db
Done.


category,number_of_invoices,invoice_percentage,total_revenue
Album,135,21.99,1718.64
Track,481,78.34,2990.79


Albums represents a smaller proportion of invoices (21.99%), but they contribute significantly to revenue.\
For this reason, the stragegies should focus on both markets: albums and tracks. Although tracks have the large proportion of transactions, albums still play an important role in the total revenue of the store.